# Day 17 — Solution: Rolling & Expanding Windows

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

## E1 — the two schemes, closed form

In [ ]:
def rolling_beta(r, m, W):
    return r.rolling(W).cov(m) / m.rolling(W).var()

def expanding_beta(r, m, min_p=60):
    return r.expanding(min_p).cov(m) / m.expanding(min_p).var()

rng = np.random.default_rng(21)
n = 3000
m = rng.normal(0.0004, 0.011, n)
y = 0.0002 + 1.10 * m + rng.normal(0, 0.009, n)
r_s, m_s = pd.Series(y), pd.Series(m)

exp_b = expanding_beta(r_s, m_s)
print(f"expanding beta: first {exp_b.dropna().iloc[0]:.3f} -> final {exp_b.iloc[-1]:.3f}  (truth 1.10)")

plt.plot(rolling_beta(r_s, m_s, 63), lw=0.6, label="W=63", alpha=0.8)
plt.plot(rolling_beta(r_s, m_s, 252), lw=1.0, label="W=252")
plt.plot(rolling_beta(r_s, m_s, 1008), lw=1.4, label="W=1008")
plt.plot(exp_b, lw=1.4, label="expanding", color="k")
plt.axhline(1.10, ls="--", c="gray", lw=0.8)
plt.legend(); plt.title("constant beta, four estimates of it"); plt.show()

**Expected pattern.** Expanding converges onto 1.10 and stays; rolling
curves orbit it with amplitude exactly inverse to √W — W = 63 ±~0.4,
W = 252 ±~0.15, W = 1008 barely breathes — while all four are estimating
THE SAME CONSTANT. Line for the log: **the schemes disagree with each
other far more than any of them disagrees with the truth; window choice
is a variance dial first, a truth dial second.**

## E2 — the wander, quantified

In [ ]:
rows = {}
sig_e, sig_x = 0.009, 0.011
for W in [63, 252, 1008]:
    rb = rolling_beta(r_s, m_s, W).dropna()
    q05, q95 = rb.quantile([0.05, 0.95])
    beta_se = sig_e / (sig_x * np.sqrt(W))
    rows[W] = [q95 - q05, 4 * beta_se, rb.mean()]
tab = pd.DataFrame(rows, index=["observed 5-95 range", "analytic ±2SE width", "mean"]).T
print(tab.round(3).to_string())

**Expected numbers (measured).** W = 63 range ≈ 0.32 (band ≈ 0.41);
W = 252 ≈ 0.17 (0.21); W = 1008 ≈ 0.06 (0.10) — wiggle tracks the band
*somewhat under* independent-window math, increasingly so at large W:
adjacent extremes share W−1 points, so the 5% and 95% quantiles of the
PATH are correlated draws and the extremes smooth toward the center (the
1.65σ-vs-2σ scaling explains most of the small-W discount; the rest is
the path being one long shared window). The sentence that pays: *at
W = 252 a constant beta still prints 5–95% ranges of ±0.10 — so a
"regime shift" inside ±0.2 at annual windows is estimation noise until
proven guilty.* And the corollary for module 13: rolling-window heatmaps
of a "time-varying effect" need, attached, the constant-world simulation
that shows the same heatmap arising from nothing.

## E3 — real rolling beta + the split test

In [ ]:
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    rr = np.log(get_prices(["AAPL", "SPY"], start="2015-01-01")).diff().dropna()
else:
    p = synthetic_prices(n_days=2500, n_assets=2, seed=23, corr=0.5)
    rr = np.log(p).diff().dropna(); rr.columns = ["AAPL", "SPY"]
r, m = rr["AAPL"], rr["SPY"]

W = 252
rb = rolling_beta(r, m, W)
rho = r.rolling(W).corr(m)
se_rb = (r.rolling(W).std() * np.sqrt(1 - rho**2)) / (m.rolling(W).std() * np.sqrt(W))
plt.plot(rb, lw=1.0, label="rolling 252d beta")
plt.fill_between(rb.index, (rb - 2*se_rb).values, (rb + 2*se_rb).values, alpha=0.25)
plt.title("AAPL rolling beta with ±2SE band"); plt.legend(); plt.show()

def ols_fit(x, y):
    x = np.asarray(x); y = np.asarray(y); nn = len(x)
    X = np.column_stack([np.ones(nn), x]); Xinv = np.linalg.inv(X.T @ X)
    b = Xinv @ X.T @ y; e = y - X @ b
    s2 = e @ e / (nn - 2)
    return b, np.sqrt(np.diag(s2 * Xinv))

split = pd.Timestamp("2020-02-01")
pre, post = rr.loc[rr.index < split], rr.loc[rr.index >= split]
b1, s1 = ols_fit(pre["SPY"], pre["AAPL"]); b2, s2 = ols_fit(post["SPY"], post["AAPL"])
z = (b2[1] - b1[1]) / np.sqrt(s1[1]**2 + s2[1]**2)
print(f"beta pre {b1[1]:.3f} (SE {s1[1]:.3f}) | post {b2[1]:.3f} (SE {s2[1]:.3f}) | z {z:+.2f}")

**Expected pattern (real AAPL).** Rolling β̂ drifts roughly 0.9–1.35 over
a decade, much of the motion **inside the band**; the split test pre/post
2020-02 gives |z| ≈ 1–2.5 depending on the name — megacap tech typically
*rose* post-2020 but by ~1–2 SEs: "suggestive drift", not "structural
break". Honest verdict sentence: **"the split point was declared by
calendar; the gap is ~1.5–2 SEs — consistent with moderate beta drift,
not proof; the risk system should blend both estimates rather than adopt
either."** Synthetic world: constant beta plant → z ≈ small |.| — the
null correctly survives.

## E4 — rolling alpha, banded

In [ ]:
am = r.rolling(W).mean() - rb * m.rolling(W).mean()     # alpha_hat per window: ȳ - βx̄
resid_std = r.rolling(W).std() * np.sqrt(1 - rho**2)
se_a = resid_std / np.sqrt(W)                            # SE of intercept approx
alpha_a = am * 252
band = 2 * se_a * 252
plt.plot(alpha_a * 100, lw=0.9, label="rolling alpha, %/yr")
plt.fill_between(alpha_a.index, (alpha_a*100 - band*100).values, (alpha_a*100 + band*100).values, alpha=0.25)
plt.axhline(0, c="k", lw=0.6); plt.title("rolling 252d alpha, annualized"); plt.legend(); plt.show()
frac_out = ((alpha_a.abs() > 2*se_a*252).mean())   # annualized on BOTH sides (watch units!)
print(f"share of window-days with |alpha| > 2SE: {frac_out:.1%}  (nominal marginal: 5%)")

**Expected pattern (measured).** Rolling alpha swings tens of %/yr while
the band runs ≈ ±30%/yr wide at 1% daily idio vol (wider for noisier
names — AAPL outside its market component lands ≈ ±40–50%/yr), and it
swallows essentially everything: share-outside ≈ 0–5% *per
realization*. Precision on why the share sits BELOW the nominal 5%: the
nominal rate is a statement about each window's marginal draw, but
adjacent windows share W−1 of W days, so the path's excursions are
correlated — a realization either clusters exits or (usually) has almost
none. **A year of daily data cannot localize a daily alpha tighter than
≈ ±σ_e·√(252/W)·2 ≈ ±30%/yr** — module 04's identity (t = SR√Y) wearing
regression clothes. Sentence for the log: "moment-to-moment 'alpha' is
mostly a sampling artifact at annual windows; only persistent multi-year
exceedance (expanding view) starts to count as evidence."

## E5 — the window answer (exemplar)

"Production hedge betas: **W = 252 daily, refreshed weekly, quoted with
the ±2SE band (±0.12 typical), with a 63-day monitor** — not for sizing
but for drift detection (if the 63-day sits outside the 252-day band for
20+ sessions, the hedge flag fires and we investigate the name, not the
window). Staleness risk: a regime shift is priced in with ~6-month lag —
accepted and compensated by the monitor; the alternative (63-day sizing)
would pay for responsiveness with SEs of ±0.25, i.e. hedge sizes set by
noise. Every hedge report carries the window rule in the footer so the
next regime break has a paper trail to blame or bless."

**Common mistakes across the day.** Reading rolling wiggle as fact;
splitting at the plot's most convenient valley (mined breakpoints);
expanding windows treated as free lunch (the faith is stability — quote
it when you spend it); and the eternal one — centered rolling anywhere
near a signal with P&L attached.